# Suivi des contrats – appareils MAX (FR)

**Ce notebook est en phase de test et propose une liste, il ne décide pas.** Avant tout déprovisionnement ou retrait de MAX, vérifier chaque appareil manuellement dans View (Opérations Services > Immeubles > Rechercher) : contrat réellement terminé et appareil encore provisionné.

**Objectif** : chaque matin ou une fois par semaine, lister les appareils FR équipés d'un MAX qui **n'ont plus de contrat actif** (à déprovisionner : MAX + SIM) ou dont **la perte est actée** (MAX à retirer).

**Lecture seule** : aucune table écrite, uniquement des vues temporaires (supprimées à la fin de la session).

**Sources**
| Table | Usage |
|---|---|
| `md.v_elevator` | Appareils, statut de provisioning, agence, MAC, ICCID |
| `md.v_contract` | Contrats et lignes d'options, statut, dates |

**Catégories**
- **À déprovisionner** : MAX provisionné, aucun contrat actif (ou aucun contrat du tout).
- **MAX à retirer** : contrat encore actif mais avec une date de résiliation renseignée (règle "fin proche" en attente de validation, voir § 5).
- **OK** : contrat actif, rien à faire.

## Constats sur les données (28/10/2026)

Ce qui a guidé les règles du notebook :

- **`ContractStatus` est fiable** : `CANCELED`, `EXPIRED`, `LOST` ont tous une date de fin dépassée ; seules 0,3 % des lignes `ACTIVE` ont une date dépassée (reconduction tacite possible ?). → On classe sur le statut, pas sur les dates.
- **`v_contract` = 1 ligne par option** : ligne principale (`ContractLine` vide) + options (dépannage, `MAX - MAX PREMIUM`, `ACGSM - LIGNE GSM`…). Une option `EXPIRED` sur un contrat `ACTIVE` est normale (environ 2 800 contrats). → Un contrat est actif si **au moins une** ligne est `ACTIVE`.
- **Historisation** : on garde la dernière version de chaque contrat (`_ChangedAtUtc` le plus récent).
- **Pas de résiliation saisie à l'avance** : aucun contrat actif n'a de `ContractCancelationdate`. Le cas 2 est vide aujourd'hui ; une date de fin proche ne suffit pas (environ 580 contrats finissent au 31/12 et sont possiblement reconduits).
- **Risque "Aucun contrat trouvé" (642 / 771)** : une partie peut être des **installations récentes** (MAX provisionné avant la saisie du contrat), pas des clients partis. À contrôler avec `v_elevator.ProvisioningStatusChangeDate` ; s'il y en a beaucoup, les signaler dans la liste.
- **Contrat suspendu (`ContractOnHold`) ≠ contrat arrêté** : il reste `ACTIVE`, l'appareil est donc « OK ». Choix assumé : on ne touche pas au MAX d'un contrat suspendu.
- **`Office` est vide en FR** → agence = `BranchNumber`.
- **`ProvisioningStatus`** : casse non homogène (`Deleted` / `deleted`) → comparaison en minuscules.
- **Connexion du MAX non disponible** : `realtime.v_elevatortwinstate` lit Cosmos DB, inaccessible depuis notre cluster, et `md.v_connectivitystatus` n'est qu'une table de libellés (Online, Offline..) sans UnitId. → Pas de colonne connexion : la vérifier dans View.

Résultat au 28/10/2026 : **8 558** appareils MAX, **771** à déprovisionner (dont **642** sans aucun contrat), **0** en cas 2.

## Paramètres

In [ ]:
# Seul endroit à modifier pour changer les règles de classement.

# Valeurs de v_elevator.ProvisioningStatus qui signifient « MAX provisionné » (comparées en minuscules).
STATUTS_PROVISIONNE = ["active", "activeerpinprogress", "updaterequired"]

# Valeurs de v_contract.ContractStatus qui signifient « sous contrat » (NOT STARTED = contrat qui va démarrer).
STATUTS_CONTRAT_ACTIF = ["ACTIVE", "NOT STARTED"]

def sql_list(values):
    """Transforme ['a', 'b'] en "'a', 'b'" pour une clause SQL IN."""
    return ", ".join(f"'{v}'" for v in values)

## 1. Appareils équipés d'un MAX

In [ ]:
# Périmètre du suivi : appareils FR dont le MAX est provisionné.
# Vue max_fr : 1 ligne par UnitId (v_elevator n'a qu'une ligne par appareil en pratique).
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW max_fr AS
SELECT
  UnitId,
  coalesce(Office, CAST(BranchNumber AS STRING)) AS agence,  -- Office est vide en FR
  ControllerType,
  ProvisioningStatus,
  MAXMacAddress          AS mac_max,   -- à relever pour le process de déprovisionnement
  MAXiccidSimCardNumber  AS iccid      -- à relever pour arrêter la SIM
FROM md.v_elevator
WHERE TenantId = 'FR'
  AND lower(ProvisioningStatus) IN ({sql_list(STATUTS_PROVISIONNE)})
""")

# Contrôle : 8 558 au 28/10/2026.
display(spark.sql("SELECT count(*) AS nb_appareils_max FROM max_fr"))

## 2. Contrat par appareil

In [ ]:
# Vue contrat_par_appareil : 1 ligne par UnitId résumant l'état de ses contrats.
# Règle : un appareil est sous contrat si au moins une ligne (principale ou option) est active
# dans la dernière version d'un de ses contrats.
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW contrat_par_appareil AS
WITH lignes AS (
  -- Lignes de contrat FR, dernière version uniquement
  SELECT
    UnitId,
    ContractNumber,
    ContractStatus,
    NextPlannedMaintenanceVisit,
    ContractStatus IN ({sql_list(STATUTS_CONTRAT_ACTIF)})  AS est_actif,
    ContractCancelationdate IS NOT NULL                    AS a_resiliation,
    least(ContractEnddate, ContractCancelationdate)        AS date_fin   -- least ignore les NULL
  FROM md.v_contract
  WHERE TenantId = 'FR'
    AND NOT coalesce(IsDeleted, false)
  QUALIFY rank() OVER (PARTITION BY UnitId, ContractNumber ORDER BY _ChangedAtUtc DESC) = 1
)
SELECT
  UnitId,
  bool_or(est_actif)                                                        AS a_contrat_actif,
  -- Contrat actif le plus long (pas de date de fin = sans fin)
  max(CASE WHEN est_actif THEN coalesce(date_fin, DATE'9999-12-31') END)    AS fin_contrat_actif,
  max_by(CASE WHEN est_actif THEN ContractNumber END,
         CASE WHEN est_actif THEN coalesce(date_fin, DATE'9999-12-31') END) AS contrat_actif,
  min(CASE WHEN est_actif AND NextPlannedMaintenanceVisit >= current_date()
           THEN NextPlannedMaintenanceVisit END)                            AS prochaine_visite,  -- peu renseigné
  bool_or(est_actif AND a_resiliation)                                      AS resiliation_notifiee,
  -- Dernier contrat terminé (utile pour le cas 1)
  max(date_fin)                                                             AS date_fin_dernier_contrat,
  max_by(ContractNumber, date_fin)                                          AS dernier_contrat,
  max_by(ContractStatus, date_fin)                                          AS statut_dernier_contrat
FROM lignes
GROUP BY UnitId
""")

## 3. Classement

In [ ]:
# Vue suivi_max : 1 ligne par appareil MAX avec sa catégorie et les infos utiles au déprovisionnement.
# LEFT JOIN : les appareils MAX sans aucun contrat sont conservés (catégorie 1, « Aucun contrat trouvé »).
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW suivi_max AS
WITH base AS (
  SELECT
    m.*,
    coalesce(c.a_contrat_actif, false) AS a_contrat_actif,
    c.UnitId IS NOT NULL               AS a_un_contrat,
    c.fin_contrat_actif, c.contrat_actif, c.prochaine_visite, c.resiliation_notifiee,
    c.date_fin_dernier_contrat, c.dernier_contrat, c.statut_dernier_contrat
  FROM max_fr m
  LEFT JOIN contrat_par_appareil c ON m.UnitId = c.UnitId
)
SELECT
  agence,
  UnitId,
  CASE
    WHEN NOT a_contrat_actif THEN 'À déprovisionner'
    WHEN coalesce(resiliation_notifiee, false) THEN 'MAX à retirer'
    -- EN ATTENTE DE VALIDATION (responsable) : retirer aussi le MAX quand la fin du contrat approche.
    -- Pour activer, décommenter la ligne suivante (60 jours = horizon à valider).
    -- Attention : les contrats reconduits tacitement (ex. fin au 31/12) remonteront aussi.
    -- WHEN datediff(fin_contrat_actif, current_date()) <= 60 THEN 'MAX à retirer'
    ELSE 'OK'
  END                                                                    AS categorie,
  CASE WHEN NOT a_un_contrat THEN 'Aucun contrat trouvé'
       WHEN NOT a_contrat_actif THEN statut_dernier_contrat
       ELSE 'ACTIVE' END                                                 AS statut_contrat,
  CASE WHEN a_contrat_actif THEN contrat_actif ELSE dernier_contrat END  AS contrat,
  CASE WHEN a_contrat_actif THEN nullif(fin_contrat_actif, DATE'9999-12-31')
       ELSE date_fin_dernier_contrat END                                 AS date_fin,
  datediff(CASE WHEN a_contrat_actif THEN nullif(fin_contrat_actif, DATE'9999-12-31')
                ELSE date_fin_dernier_contrat END, current_date())       AS jours_avant_fin,  -- négatif = déjà fini
  coalesce(resiliation_notifiee, false)                                  AS resiliation_notifiee,
  prochaine_visite,
  ProvisioningStatus,
  ControllerType,
  mac_max,
  iccid
FROM base
""")

# Synthèse du jour : nombre d'appareils par catégorie.
display(spark.sql("""
SELECT categorie, count(*) AS nb_appareils
FROM suivi_max
GROUP BY categorie
ORDER BY categorie
"""))

## 4. Cas 1 – À déprovisionner (MAX + SIM)

Suivre ensuite le process de déprovisionnement (DSC) et relever MAC + ICCID.

In [ ]:
# Liste du cas 1. Ordre : contrats terminés depuis le plus longtemps,
# puis les « Aucun contrat trouvé » (sans date) en fin de liste : à vérifier en priorité dans View.
display(spark.sql("""
SELECT * FROM suivi_max
WHERE categorie = 'À déprovisionner'
ORDER BY jours_avant_fin NULLS LAST, agence
"""))

## 5. Cas 2 – MAX à retirer avant la fin du contrat

Vide tant que les résiliations ne sont pas saisies à l'avance dans `v_contract`. Se remplira automatiquement sinon.

Règle « fin de contrat proche » (≤ 60 jours) prête mais **désactivée** en § 3, en attente de validation par le responsable : elle ferait aussi remonter les contrats reconduits tacitement.

In [ ]:
# Liste du cas 2, par date de fin la plus proche.
# Idéalement, retirer le MAX lors de la prochaine visite d'entretien pour éviter un déplacement.
display(spark.sql("""
SELECT * FROM suivi_max
WHERE categorie = 'MAX à retirer'
ORDER BY jours_avant_fin, agence
"""))

## 6. Contrôle

In [ ]:
# Les trois valeurs doivent être égales : 1 ligne par appareil MAX, aucun appareil perdu ni dupliqué.
display(spark.sql("""
SELECT
  (SELECT count(*) FROM max_fr)  AS nb_appareils_max,
  count(*)                       AS nb_lignes_suivi,
  count(DISTINCT UnitId)         AS nb_unitid_suivi
FROM suivi_max
"""))

## Planification

*Workflows → Jobs → Create job* → tâche de type **Notebook** → ce notebook → **Schedule** tous les jours à 7h00 (Europe/Paris) ou une fois par semaine → *Notifications → On failure* : e-mail du responsable.

Le résultat de chaque exécution reste consultable dans l'onglet **Runs** du job : il sert d'historique sans écrire de table. On pourra enregistrer les résultats une fois les droits obtenus.

## Idée pour plus tard : option MAX absente du contrat

`v_contract` contient des lignes d'options dont `MAX - MAX PREMIUM`, `MAX1 - MAX PLUS` et `ACGSM - ABONNEMENT ET COMM - LIGNE GSM`.

Un appareil peut avoir un contrat actif **sans option MAX active** : le client ne paie plus le MAX mais le boîtier reste provisionné. Le repérer (filtre sur `ContractLine LIKE 'MAX%'`) donnerait une 4e catégorie, et sans doute le vrai "cas 2" que les dates de résiliation ne permettent pas de détecter.

**Rappel phase de test : vérifier manuellement chaque appareil dans View avant de déprovisionner**, en particulier ceux marqués « Aucun contrat trouvé » (installation récente, contrat sous un autre UnitId…). Un déprovisionnement à tort coupe le service d'un client sous contrat.